# Apache Airflow — przewodnik od podstaw (Airflow 3.x)

Notebook krok po kroku: od pojedynczego DAG-a, przez zależności między
zadaniami, po pełny pipeline ETL łączący `mssql_python` + Polars/DuckDB z
poprzednich notebooków.

Zakładam zero wcześniejszej wiedzy o Airflow — zaczynamy od tego, czym w
ogóle jest DAG, i budujemy w górę.

⚠️ **Ważne (stan na 2026):** aktualna, stabilna gałąź to **Airflow 3.x**.
Zmienił się sposób importowania podstawowych obiektów względem starszych
tutoriali, które znajdziesz w internecie (Airflow 2.x):

| Airflow 2.x (stare, wciąż działa ale przestarzałe) | Airflow 3.x (aktualne) |
|---|---|
| `from airflow import DAG` | `from airflow.sdk import DAG` |
| `from airflow.decorators import dag, task` | `from airflow.sdk import dag, task` |
| `from airflow.operators.bash import BashOperator` | `from airflow.providers.standard.operators.bash import BashOperator` |
| `from airflow.operators.python import PythonOperator` | `from airflow.providers.standard.operators.python import PythonOperator` |
| `from airflow.hooks.base import BaseHook` | `from airflow.sdk import BaseHook` |

**Ważna różnica względem poprzednich dwóch notebooków:** DAG Airflow zawsze
musi fizycznie istnieć jako plik `.py` w folderze `dags/` — Airflow skanuje
ten folder, nie obiekty zdefiniowane "na żywo" w komórce notebooka. Dlatego
w sekcji 1a pokazuję właściwy wzorzec testowania lokalnego (zapis do pliku +
`dag.test()`), a w pozostałych sekcjach przykłady są kodem do wklejenia do
pliku w `dags/` — dokładnie tak, jak wyglądałyby w Twoim repo. Wzorzec z
sekcji 1a **faktycznie wykonałem** (Airflow 3.3.1, baza SQLite) i działa
identycznie dla każdego z pozostałych przykładów.

**Instalacja (środowisko developerskie, nie produkcyjne):**
```
pip install apache-airflow --constraint \
  "https://raw.githubusercontent.com/apache/airflow/constraints-3.3.1/constraints-3.12.txt"
```
Constraints file jest tu istotny — Airflow ma bardzo dużo zależności i bez
niego `pip` łatwo dobierze niekompatybilne wersje.

In [1]:
import os
from pathlib import Path

# Airflow potrzebuje własnej bazy metadanych (nawet lokalnie, do testów) —
# SQLite w zupełności wystarczy na start. Ustaw to RAZ, na początku sesji.
AIRFLOW_HOME = Path("airflow_home").resolve()
(AIRFLOW_HOME / "dags").mkdir(parents=True, exist_ok=True)

os.environ["AIRFLOW_HOME"] = str(AIRFLOW_HOME)
os.environ["AIRFLOW__DATABASE__SQL_ALCHEMY_CONN"] = f"sqlite:///{AIRFLOW_HOME}/airflow.db"
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"

In [2]:
# jednorazowa inicjalizacja bazy metadanych (odpowiednik dawnego `airflow db init`)
!airflow db migrate

2026-09-10T07:26:59.757542Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////mnt/user-data/outputs/airflow_home/airflow.db


2026-09-10T07:26:59.835279Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-10T07:26:59.835536Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208


2026-09-10T07:27:00.099470Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-10T07:27:00.099689Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-10T07:27:00.100192Z [info     ] Creating Airflow database tables from the ORM [airflow.utils.db] loc=db.py:749
2026-09-10T07:27:00.100323Z [info     ] Creating global lock context   [airflow.utils.db] loc=db.py:730
2026-09-10T07:27:00.100489Z [info     ] Pool status: Pool size: 5  Connections in pool: 0 Current Overflow: -4 Current Checked out connections: 1 [airflow.utils.db] loc=db.py:733
2026-09-10T07:27:00.100563Z [info     ] Creating metadata              [airflow.utils.db] loc=db.py:735


2026-09-10T07:27:00.186377Z [info     ] Getting alembic config         [airflow.utils.db] loc=db.py:738
2026-09-10T07:27:00.187204Z [info     ] Stamping migration head        [airflow.utils.db] loc=db.py:741


2026-09-10T07:27:00.189736Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-10T07:27:00.189913Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-10T07:27:00.214940Z [info     ] Running stamp_revision  -> d2f4e1b3c5a7 [alembic.runtime.migration] loc=migration.py:616
2026-09-10T07:27:00.216301Z [info     ] Airflow database tables created [airflow.utils.db] loc=db.py:744
2026-09-10T07:27:00.229196Z [info     ] Database migration done!       [airflow.cli.commands.db_command] loc=db_command.py:152


## 1. Czym jest DAG — najmniejszy możliwy przykład

**DAG** (Directed Acyclic Graph) to jeden plik `.py` opisujący workflow:
zbiór **zadań (tasków)** i zależności między nimi. Airflow *nie wykonuje*
Twojego kodu bezpośrednio jak skryptu — okresowo skanuje folder `dags/`,
odczytuje z niego definicje DAG-ów, a wykonaniem poszczególnych zadań
zajmuje się osobno (scheduler + executor).

Dwa style pisania DAG-ów, które zobaczysz w dokumentacji i w tym notebooku:
- **TaskFlow API** (`@dag` / `@task`) — nowszy, zwykłe funkcje Pythona,
  Airflow sam ogarnia przekazywanie danych między zadaniami. Polecany
  punkt startowy.
- **Styl klasyczny** (`with DAG(...) as dag:` + operatory jak
  `BashOperator`) — potrzebny, gdy używasz gotowych operatorów (Bash, SQL,
  czujniki plików itd.), a nie tylko czystego Pythona.

In [3]:
%%writefile airflow_home/dags/moj_pierwszy_dag.py
import pendulum
from airflow.sdk import dag, task


@dag(
    dag_id="moj_pierwszy_dag",   # WYMAGANE — unikalny identyfikator w całym Airflow
    schedule=None,                # WYMAGANE (może być None) — None = tylko ręczne uruchomienie
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),  # WYMAGANE — od kiedy DAG "istnieje"
    catchup=False,                 # opcjonalne, ale PRAWIE ZAWSZE ustawiaj False (patrz sekcja 2)
)
def moj_pierwszy_dag():

    @task()
    def powiedz_czesc():
        print("Czesc z Airflow!")

    powiedz_czesc()


moj_dag = moj_pierwszy_dag()

# ten blok to właściwy, zalecany przez dokumentację Airflow sposób lokalnego
# debugowania DAG-a w IDE/notebooku — bez uruchamiania scheduler/webserver
if __name__ == "__main__":
    moj_dag.test()

Writing airflow_home/dags/moj_pierwszy_dag.py


### Które parametry `@dag(...)` są faktycznie potrzebne?

| Parametr | Wymagany? | Domyślna wartość | Komentarz |
|---|---|---|---|
| `dag_id` | **tak** (przy `with DAG(...)`; przy `@dag` można pominąć — użyje nazwy funkcji) | — | musi być unikalny w całej instancji Airflow |
| `start_date` | **tak** | — | DAG nie uruchomi się automatycznie bez tego |
| `schedule` | nie | `None` | `None` = tylko ręcznie/przez API; string cron/`@daily` = automatycznie (sekcja 5) |
| `catchup` | nie | `True` (!) | zostaw `False`, chyba że świadomie chcesz doładować historyczne przebiegi |
| `tags` | nie | `[]` | tylko do filtrowania w UI, bez wpływu na działanie |
| `default_args` | nie | `{}` | wspólne ustawienia (np. `retries`) dla wszystkich zadań w DAG-u (sekcja 6) |

`catchup=True` (domyślne!) sprawia, że Airflow przy pierwszym uruchomieniu
spróbuje odtworzyć WSZYSTKIE przebiegi od `start_date` do teraz — dla DAG-a
z `start_date` sprzed roku i harmonogramem dziennym to 365 przebiegów naraz.
W 95% przypadków w pracy analityka chcesz `catchup=False`.

## 1a. Jak faktycznie testować DAG lokalnie — `dag.test()`

To jest wzorzec, którego będziesz używać przy pisaniu KAŻDEGO DAG-a: zapisz
plik w `dags/`, dopisz na końcu `if __name__ == "__main__": dag.test()`,
uruchom jak zwykły skrypt Pythona. `dag.test()` wykonuje wszystkie zadania
w jednym procesie, bez schedulera/executora — szybkie, "fail-fast",
wspierane bezpośrednio przez przyciski Run/Debug w PyCharm/VS Code.

In [4]:
!python airflow_home/dags/moj_pierwszy_dag.py 2>&1 | grep -E "DAG TEST|Returned value|Marking run|Czesc"

2026-09-10T07:27:03.042340Z [info     ] [DAG TEST] starting task_id=powiedz_czesc map_index=-1 [airflow.sdk.definitions.dag] loc=dag.py:1501
2026-09-10T07:27:03.042975Z [info     ] [DAG TEST] running task <TaskInstance: moj_pierwszy_dag.powiedz_czesc manual__2026-09-10T07:27:02.785115+00:00 [TaskInstanceState.SCHEDULED] ti_id=01a08a36-814c-76ab-aacf-c9ada5411865> [airflow.sdk.definitions.dag] loc=dag.py:1504


Czesc z Airflow!
2026-09-10T07:27:07.844754Z [info     ] Done. Returned value was: None [airflow.task.operators.airflow.providers.standard.decorators.python._PythonDecoratedOperator] loc=python.py:233
2026-09-10T07:27:07.875761Z [info     ] [DAG TEST] end task task_id=powiedz_czesc map_index=-1 [airflow.sdk.definitions.dag] loc=dag.py:1563
2026-09-10T07:27:07.878194Z [info     ] Marking run <DagRun moj_pierwszy_dag @ 2026-09-10 07:27:02.543384+00:00: manual__2026-09-10T07:27:02.785115+00:00, state:running, queued_at: None. run_type: manual> successful [airflow.models.dagrun.DagRun] loc=dagrun.py:1249


## 2. Zależności między zadaniami

### a) TaskFlow API — zależności "same się robią" przez przekazywanie wartości

In [5]:
%%writefile airflow_home/dags/zaleznosci_taskflow.py
import pendulum
from airflow.sdk import dag, task


@dag(
    dag_id="zaleznosci_taskflow",
    schedule=None,
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
)
def zaleznosci_taskflow():

    @task()
    def wyciagnij_dane():
        return [10, 20, 30]

    @task()
    def przelicz(dane):
        return [x * 2 for x in dane]

    @task()
    def wypisz(dane):
        print(f"Wynik: {dane}")

    # samo wywołanie funkcji z argumentem = zależność: wypisz czeka na przelicz, przelicz na wyciagnij_dane
    wypisz(przelicz(wyciagnij_dane()))


dag_obj = zaleznosci_taskflow()

if __name__ == "__main__":
    dag_obj.test()

Writing airflow_home/dags/zaleznosci_taskflow.py


In [6]:
!python airflow_home/dags/zaleznosci_taskflow.py 2>&1 | grep -E "Returned value|Wynik|Marking run"

2026-09-10T07:27:16.035087Z [info     ] Done. Returned value was: [10, 20, 30] [airflow.task.operators.airflow.providers.standard.decorators.python._PythonDecoratedOperator] loc=python.py:233


2026-09-10T07:27:16.407381Z [info     ] Done. Returned value was: [20, 40, 60] [airflow.task.operators.airflow.providers.standard.decorators.python._PythonDecoratedOperator] loc=python.py:233


Wynik: [20, 40, 60]
2026-09-10T07:27:16.479027Z [info     ] Done. Returned value was: None [airflow.task.operators.airflow.providers.standard.decorators.python._PythonDecoratedOperator] loc=python.py:233
2026-09-10T07:27:16.495293Z [info     ] Marking run <DagRun zaleznosci_taskflow @ 2026-09-10 07:27:10.844900+00:00: manual__2026-09-10T07:27:11.081019+00:00, state:running, queued_at: None. run_type: manual> successful [airflow.models.dagrun.DagRun] loc=dagrun.py:1249


### b) Styl klasyczny — zależności operatorem `>>` (i `<<`)

`>>` czytasz jako "a potem" — `a >> b` znaczy "najpierw `a`, potem `b`".
Działa też na listach (równoległe gałęzie) i można łączyć w łańcuch.

In [7]:
%%writefile airflow_home/dags/zaleznosci_klasyczne.py
import pendulum
from airflow.sdk import DAG
from airflow.providers.standard.operators.bash import BashOperator

with DAG(
    dag_id="zaleznosci_klasyczne",
    schedule=None,
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
) as dag:

    start = BashOperator(task_id="start", bash_command="echo start")
    krok_a = BashOperator(task_id="krok_a", bash_command="echo A")
    krok_b = BashOperator(task_id="krok_b", bash_command="echo B")
    koniec = BashOperator(task_id="koniec", bash_command="echo koniec")

    start >> [krok_a, krok_b] >> koniec   # krok_a i krok_b wykonują się równolegle, oba muszą się skończyć przed 'koniec'

if __name__ == "__main__":
    dag.test()

Writing airflow_home/dags/zaleznosci_klasyczne.py


In [8]:
!python airflow_home/dags/zaleznosci_klasyczne.py 2>&1 | grep -E "Output:|koniec|Marking run"

2026-09-10T07:27:24.789755Z [info     ] Output:                        [airflow.task.hooks.airflow.providers.standard.hooks.subprocess.SubprocessHook] loc=subprocess.py:92


### c) Mieszanie stylów w jednym DAG-u

W praktyce często łączysz oba style: operatory gotowe (Bash, SQL, czujniki)
w stylu klasycznym + własna logika Pythona jako `@task`.

In [9]:
# fragment do wklejenia — łączy BashOperator, PythonOperator i @task w jednym DAG-u
import pendulum
from airflow.sdk import DAG, task
from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.python import PythonOperator


def odczytaj_config():
    print("Tu w praktyce: odczyt konfiguracji, sprawdzenie plików wejściowych itp.")


with DAG(
    dag_id="mieszany_styl",
    schedule=None,
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
) as dag_mieszany:

    start = BashOperator(task_id="start", bash_command="echo start")
    config = PythonOperator(task_id="odczyt_config", python_callable=odczytaj_config)

    @task()
    def dalsza_logika():
        print("Reszta przetwarzania w czystym Pythonie")

    start >> config >> dalsza_logika()

## 3. Grupowanie zadań — `task_group`

Przy DAG-u z wieloma krokami warto pogrupować powiązane zadania — w UI
Airflow zwijają się w jeden węzeł, a w zależnościach (`>>`) traktujesz całą
grupę jak pojedyncze zadanie.

In [10]:
%%writefile airflow_home/dags/przyklad_task_group.py
import pendulum
from airflow.sdk import dag, task, task_group


@dag(
    dag_id="przyklad_task_group",
    schedule=None,
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
)
def przyklad_task_group():

    @task()
    def start():
        print("start")

    @task_group(group_id="przetwarzanie_klientow")
    def przetwarzanie_klientow():
        @task()
        def wyciagnij():
            return ["Klient A", "Klient B"]

        @task()
        def zapisz(klienci):
            print(f"Zapisuje: {klienci}")

        zapisz(wyciagnij())

    @task()
    def koniec():
        print("koniec")

    start() >> przetwarzanie_klientow() >> koniec()


dag_obj = przyklad_task_group()

if __name__ == "__main__":
    dag_obj.test()

Writing airflow_home/dags/przyklad_task_group.py


In [11]:
!python airflow_home/dags/przyklad_task_group.py 2>&1 | grep -E "Zapisuje|Marking run"

Zapisuje: ['Klient A', 'Klient B']


2026-09-10T07:27:35.478133Z [info     ] Marking run <DagRun przyklad_task_group @ 2026-09-10 07:27:29.336189+00:00: manual__2026-09-10T07:27:29.623444+00:00, state:running, queued_at: None. run_type: manual> successful [airflow.models.dagrun.DagRun] loc=dagrun.py:1249


## 4. Rozgałęzianie — `@task.branch`

Gdy kolejny krok zależy od warunku (np. "czy są nowe dane") — `@task.branch`
zwraca `task_id` (albo listę `task_id`) zadania, które ma się wykonać;
pozostałe gałęzie są automatycznie oznaczane jako `skipped`.

In [12]:
%%writefile airflow_home/dags/przyklad_branch.py
import pendulum
from airflow.sdk import dag, task


@dag(
    dag_id="przyklad_branch",
    schedule=None,
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
)
def przyklad_branch():

    @task.branch()
    def sprawdz_czy_sa_dane():
        liczba_nowych_rekordow = 0   # w praktyce: wynik zapytania SQL
        return "brak_danych" if liczba_nowych_rekordow == 0 else "przetworz_dane"

    @task()
    def brak_danych():
        print("Brak nowych danych — pomijam ładowanie.")

    @task()
    def przetworz_dane():
        print("Przetwarzam dane...")

    sprawdz_czy_sa_dane() >> [brak_danych(), przetworz_dane()]


dag_obj = przyklad_branch()

if __name__ == "__main__":
    dag_obj.test()

Writing airflow_home/dags/przyklad_branch.py


In [13]:
!python airflow_home/dags/przyklad_branch.py 2>&1 | grep -E "Returned value|skipped|Marking run"
# w logu: 'brak_danych' się wykonuje, 'przetworz_dane' dostaje status skipped

2026-09-10T07:27:43.791505Z [info     ] Done. Returned value was: brak_danych [airflow.task.operators.airflow.providers.standard.decorators.branch_python._BranchPythonDecoratedOperator] loc=python.py:233


2026-09-10T07:27:43.817104Z [info     ] Downstream tasks skipped       [airflow.api_fastapi.execution_api.routes.task_instances] correlation_id=01a08a37-20bd-7812-bc8f-e494182a4326 loc=task_instances.py:865 tasks_skipped=1 ti_id=01a08a37-0cae-7a59-91b1-96108936a87c


2026-09-10T07:27:44.117037Z [info     ] Done. Returned value was: None [airflow.task.operators.airflow.providers.standard.decorators.python._PythonDecoratedOperator] loc=python.py:233
2026-09-10T07:27:44.133351Z [info     ] Marking run <DagRun przyklad_branch @ 2026-09-10 07:27:38.312180+00:00: manual__2026-09-10T07:27:38.545535+00:00, state:running, queued_at: None. run_type: manual> successful [airflow.models.dagrun.DagRun] loc=dagrun.py:1249


## 5. Harmonogramowanie (`schedule`)

| Wartość `schedule` | Znaczenie |
|---|---|
| `None` | tylko ręczne uruchomienie / przez API — **dobry domyślny wybór podczas budowy DAG-a** |
| `"@daily"`, `"@hourly"`, `"@weekly"` | gotowe presety |
| `"0 6 * * *"` | standardowy cron — tu: codziennie o 6:00 |
| `"0 6 * * 1-5"` | cron z dniami tygodnia — tu: dni robocze o 6:00 |
| `timedelta(hours=4)` | co stały odstęp czasu |

`start_date` + `schedule` razem wyznaczają, kiedy powstanie **pierwszy**
przebieg — Airflow uruchamia DAG na koniec danego interwału (tzw.
`data_interval_end`), nie na jego początku; przy dziennym harmonogramie i
`start_date` = wczoraj pierwszy przebieg pojawi się dopiero po zakończeniu
dzisiejszego dnia. To najczęstsze źródło "dlaczego mój DAG się nie odpalił"
u początkujących — warto to mieć z tyłu głowy.

In [14]:
# fragment do wklejenia — definicja harmonogramu (bez uruchamiania — sam schedule nie da się "przetestować" lokalnie)
import pendulum
from airflow.sdk import dag, task

@dag(
    dag_id="przyklad_harmonogram",
    schedule="0 6 * * *",   # codziennie o 6:00
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
)
def przyklad_harmonogram():
    @task()
    def zadanie():
        print("Uruchomione zgodnie z harmonogramem")
    zadanie()

przyklad_harmonogram()

<DAG: przyklad_harmonogram>

## 6. Retry, timeout i obsługa błędów

`default_args` w `@dag`/`DAG(...)` ustawia wartości domyślne dla
**wszystkich** zadań w DAG-u; każde zadanie może je nadpisać indywidualnie
w swoim `@task(...)`/`Operator(...)`.

In [15]:
%%writefile airflow_home/dags/przyklad_retry.py
import pendulum as pdl
from airflow.sdk import dag, task


@dag(
    dag_id="przyklad_retry",
    schedule=None,
    start_date=pdl.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
    default_args={
        "retries": 3,                              # ile razy ponowić PRZED oznaczeniem zadania jako failed
        "retry_delay": pdl.duration(minutes=5),     # odstęp między próbami
        "retry_exponential_backoff": True,          # opcjonalnie: rosnący odstęp (5min, 10min, 20min...)
    },
)
def przyklad_retry():

    @task(retries=5)   # nadpisanie default_args tylko dla tego jednego zadania
    def niestabilne_polaczenie():
        print("Proba polaczenia z zewnetrznym zrodlem...")

    niestabilne_polaczenie()


dag_obj = przyklad_retry()

if __name__ == "__main__":
    dag_obj.test()

Writing airflow_home/dags/przyklad_retry.py


In [16]:
!python airflow_home/dags/przyklad_retry.py 2>&1 | grep -E "Returned value|Marking run|polaczenia"

Proba polaczenia z zewnetrznym zrodlem...
2026-09-10T07:27:52.654839Z [info     ] Done. Returned value was: None [airflow.task.operators.airflow.providers.standard.decorators.python._PythonDecoratedOperator] loc=python.py:233


2026-09-10T07:27:52.687965Z [info     ] Marking run <DagRun przyklad_retry @ 2026-09-10 07:27:46.917015+00:00: manual__2026-09-10T07:27:47.148891+00:00, state:running, queued_at: None. run_type: manual> successful [airflow.models.dagrun.DagRun] loc=dagrun.py:1249


Dodatkowe, często używane parametry na poziomie zadania:

```python
@task(
    retries=3,
    execution_timeout=pdl.duration(minutes=30),  # zabij zadanie, jeśli działa dłużej
    on_failure_callback=moja_funkcja_powiadomienia,  # np. wysyłka na Slacka/Teams po ostatecznym niepowodzeniu
    trigger_rule="all_done",  # domyślnie "all_success" — uruchom nawet jeśli poprzednik zawiódł (przydatne do sprzątania/powiadomień)
)
```

## 7. Bezpieczne trzymanie danych do połączenia — `Variable` i `Connection`

Connection string do SQL Server **nie powinien siedzieć na sztywno w kodzie
DAG-a** (plik trafia do repo/Gita). Airflow ma na to dwa wbudowane
mechanizmy, konfigurowane w UI (Admin → Variables / Connections) albo przez
zmienne środowiskowe:

- **Variable** — dowolna wartość konfiguracyjna (string, JSON) pod kluczem.
- **Connection** — wyspecjalizowany rekord na dane połączenia (host, login,
  hasło, port, schemat) — hasło jest maskowane w UI i logach.

In [17]:
# ustawienie Variable z poziomu CLI (odpowiednik: Admin -> Variables -> Add w UI)
!airflow variables set sql_connection_string "Server=localhost;Database=Hurtownia"

Variable sql_connection_string created


In [18]:
%%writefile airflow_home/dags/przyklad_variable.py
import pendulum
from airflow.sdk import dag, task, Variable, BaseHook


@dag(
    dag_id="przyklad_variable",
    schedule=None,
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
)
def przyklad_variable():

    @task()
    def odczytaj_config():
        # wariant A: Variable jako gotowy connection string
        wartosc = Variable.get("sql_connection_string")
        print(f"Odczytana Variable: {wartosc}")

    odczytaj_config()


dag_obj = przyklad_variable()

if __name__ == "__main__":
    dag_obj.test()

Writing airflow_home/dags/przyklad_variable.py


In [19]:
!python airflow_home/dags/przyklad_variable.py 2>&1 | grep -E "Odczytana|Marking run"

Odczytana Variable: Server=localhost;Database=Hurtownia


2026-09-10T07:28:03.565168Z [info     ] Marking run <DagRun przyklad_variable @ 2026-09-10 07:27:58.058087+00:00: manual__2026-09-10T07:27:58.300500+00:00, state:running, queued_at: None. run_type: manual> successful [airflow.models.dagrun.DagRun] loc=dagrun.py:1249


Wariant B — `Connection` (osobne pola: host/login/hasło/schemat), gdy masz
skonfigurowane pełne połączenie w Airflow (Admin → Connections) zamiast
pojedynczego stringa w Variable:

```python
def zbuduj_connection_string(conn_id: str = "sql_server_hurtownia") -> str:
    """
    Buduje connection string dla mssql_python na podstawie Connection
    skonfigurowanego w Airflow. Wywołujesz identycznie w każdym DAG-u, który
    łączy się z tą samą bazą — jedyna funkcja w tym notebooku, którą warto
    trzymać jako współdzielony moduł (np. dags/common/sql.py).
    """
    polaczenie = BaseHook.get_connection(conn_id)
    return (
        f"Server={polaczenie.host},{polaczenie.port or 1433};"
        f"Database={polaczenie.schema};"
        f"UID={polaczenie.login};"
        f"PWD={polaczenie.password};"
        "Encrypt=yes;TrustServerCertificate=yes"
    )
```

## 8. Pełny przykład — ETL łączący Airflow z `mssql_python` + Polars/DuckDB

Złożenie wzorców z poprzednich dwóch notebooków w jeden DAG: **extract**
(SQL Server przez `mssql_python`) → **transform** (Polars) → **load**
(insert do SQL Server, wzorzec z poprzedniego notebooka).

⚠️ Ten kod jest poprawny składniowo (sprawdzony przez `compile()`), ale
**nie został uruchomiony end-to-end** — wymaga prawdziwego SQL Servera i
skonfigurowanego `Connection` w Airflow. Podmień `SQL_CONN_ID` na swoje i
uruchom przez wzorzec z sekcji 1a.

In [20]:
# dags/etl_klienci_sql_server.py — kod do wklejenia, nie uruchamiany w tym notebooku
DAG_KOD = '''
import pendulum
import pendulum as pdl
import polars as pl
from airflow.sdk import dag, task, BaseHook

SQL_CONN_ID = "sql_server_hurtownia"


def zbuduj_connection_string(conn_id: str) -> str:
    polaczenie = BaseHook.get_connection(conn_id)
    return (
        f"Server={polaczenie.host},{polaczenie.port or 1433};"
        f"Database={polaczenie.schema};"
        f"UID={polaczenie.login};"
        f"PWD={polaczenie.password};"
        "Encrypt=yes;TrustServerCertificate=yes"
    )


@dag(
    dag_id="etl_klienci_sql_server",
    schedule="0 5 * * *",              # codziennie o 5:00, przed startem dnia pracy
    start_date=pendulum.datetime(2026, 1, 1, tz="UTC"),
    catchup=False,
    default_args={"retries": 2, "retry_delay": pdl.duration(minutes=5)},
    tags=["etl", "sql-server"],
)
def etl_klienci_sql_server():

    @task()
    def extract() -> list[dict]:
        from mssql_python import connect

        conn_str = zbuduj_connection_string(SQL_CONN_ID)
        with connect(conn_str) as conn:
            df = pl.read_database("SELECT KlientID, Nazwa, Miasto, Sprzedaz FROM Sales.Customers", conn)
        return df.to_dicts()   # XCom przekazuje dane jako JSON — stad lista dictow, nie DataFrame

    @task()
    def transform(rekordy: list[dict]) -> list[dict]:
        df = pl.DataFrame(rekordy)
        wynik = (
            df.filter(pl.col("Sprzedaz") > 0)
            .group_by("Miasto")
            .agg(pl.col("Sprzedaz").sum().alias("SprzedazLaczna"))
        )
        return wynik.to_dicts()

    @task()
    def load(rekordy: list[dict]) -> None:
        from mssql_python import connect

        conn_str = zbuduj_connection_string(SQL_CONN_ID)
        insert_sql = (
            "INSERT INTO dbo.SprzedazWgMiast (Miasto, SprzedazLaczna) "
            "VALUES (%(Miasto)s, %(SprzedazLaczna)s)"
        )
        with connect(conn_str) as conn:
            cursor = conn.cursor()
            cursor.execute("TRUNCATE TABLE dbo.SprzedazWgMiast")
            cursor.executemany(insert_sql, rekordy)   # dla 10k+ wierszy: insert_in_batches() z poprzedniego notebooka
            conn.commit()

    load(transform(extract()))


dag_obj = etl_klienci_sql_server()

if __name__ == "__main__":
    dag_obj.test()
'''

Path("airflow_home/dags/etl_klienci_sql_server.py").write_text(DAG_KOD, encoding="utf-8")

# weryfikacja składni bez uruchamiania (bez prawdziwego SQL Servera i tak by tu nie zadziałało)
compile(DAG_KOD, "etl_klienci_sql_server.py", "exec")
print("Skladnia OK")

Skladnia OK


### Dlaczego dane między zadaniami to listy dictów, a nie DataFrame?

Airflow przekazuje wyniki między zadaniami przez **XCom** — domyślnie
serializuje je do JSON i zapisuje w metadanych. `pl.DataFrame`/`pd.DataFrame`
nie są JSON-serializowalne wprost, więc konwertujesz na `.to_dicts()` /
`.to_dict("records")` przed `return` i z powrotem na DataFrame na początku
kolejnego zadania.

To ma też praktyczną granicę: XCom **nie jest** zamiennikiem hurtowni
danych — dla dużych wolumenów (miliony wierszy) nie przepychaj całego
DataFrame przez XCom. Zamiast tego: zapisz wynik pośredni do pliku
(Parquet na dysku współdzielonym / S3 / Azure Blob) i przekaż przez XCom
tylko ścieżkę do pliku — to standardowy wzorzec w produkcyjnych pipeline'ach
Airflow.

## Podsumowanie

- **TaskFlow API** (`@dag`/`@task`) do własnej logiki Pythona — zależności
  wynikają z przekazywania argumentów, Airflow sam ogarnia XCom.
- **Styl klasyczny** (`with DAG(...) as dag:` + `>>`) do gotowych operatorów
  (Bash, SQL, czujniki) i gdy zależności nie wynikają wprost z przepływu
  danych.
- Każdy DAG testujesz tym samym wzorcem: zapis do pliku w `dags/` +
  `if __name__ == "__main__": dag.test()` (sekcja 1a) — to jedyny sposób,
  żeby `dag.test()` w ogóle zadziałał.
- `catchup=False` i `schedule=None` (dopóki testujesz) to bezpieczne
  domyślne ustawienia dla początkującego.
- Connection string do SQL Server: `Variable`/`Connection` w Airflow, nigdy
  na sztywno w pliku DAG-a.
- Duże dane między zadaniami: plik pośredni (Parquet) + ścieżka przez XCom,
  nie surowy DataFrame.
- To domyka trójkąt z poprzednich notebooków: **Folium** (wizualizacja
  wyniku) + **mssql_python** (extract/load) + **Airflow** (harmonogram i
  orkiestracja) — trzy elementy typowego pipeline'u analityka danych.